# ContractScanner LLM Evaluation, MLflow Tracing, and ROI Comparison

This notebook adds the final LLM and evaluation layer for the ContractScanner AI Agent final project.

The previous notebook prepared the contract analysis examples by loading CUAD contract chunks, retrieving relevant clauses, checking scope, and assigning a basic risk level. This notebook builds on those saved outputs by:

- Running contract questions through GPT-4o and GPT-4o-mini
- Keeping the out-of-scope examples as graceful rejection cases
- Logging five end-to-end examples with MLflow
- Creating evaluation scores for each example
- Explaining how human review fits into the evaluation process
- Comparing GPT-4o and GPT-4o-mini from a business ROI perspective

The goal is to show not only that the agent works technically, but also that it can be evaluated and compared in a way that makes sense for an enterprise legal/compliance use case.

In [0]:
# Install the packages needed for this notebook.
# openai is used for GPT-4o and GPT-4o-mini calls.
# mlflow is used for trace/evaluation logging.
# pandas is used to load and organize the saved agent outputs.

%pip install -q openai mlflow pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install -U mlflow
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
# Import the Python libraries used throughout this notebook.

import os
import pandas as pd
import time
import mlflow
from openai import OpenAI

In [0]:
# Check the current notebook working directory.
# This helps confirm that Databricks is looking in the correct repo folder.

print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

Current folder: /Workspace/Users/shifrin.robert@gmail.com/contractscanner-ai-agent/notebooks
Files here: ['agent_outputs', 'CUAD_Data_Pipeline.ipynb', '02_agent_and_tools.ipynb.ipynb', 'cuad_chunks.pkl', 'cuad_faiss_index.index', 'cuad_embeddings.npy', '03_llm_evaluation_and_roi.ipynb', '01_data_pipeline.ipynb', '.gitkeep']


In [0]:
# These are the saved outputs from the previous agent/tool prep notebook.
# The first file contains the 3 contract questions with retrieved contract context.
# The second file contains the 2 graceful rejection examples.

prep_path = "agent_outputs/agent_context_prep_results.csv"
reject_path = "agent_outputs/rejection_examples.csv"

print("Prep exists:", os.path.exists(prep_path))
print("Reject exists:", os.path.exists(reject_path))

Prep exists: True
Reject exists: True


In [0]:
# Load the saved examples into pandas DataFrames so we can inspect and reuse them.

prep_df = pd.read_csv(prep_path)
reject_df = pd.read_csv(reject_path)

display(prep_df)
display(reject_df)

question,status,risk_level,latency_seconds,top_retrieved_clause
Does this contract contain risky termination clauses?,ready_for_llm,Medium,0.197,"32.4. the following Clauses shall survive termination for whatever cause of this Agreement: Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34 inclusive."
What indemnification obligations are present?,ready_for_llm,High,0.037,"The foregoing indemnification obligations are conditioned upon the indemnified party promptly notifying the indemnifying party in writing of the claim, suit or proceeding for which the indemnifying party is obligated under this Section 24, cooperating with, assisting and providing information to, the indemnifying party as reasonably required, and granting the indemnifying party the exclusive right to defend or settle such claim, suit or proceeding."
Summarize the key risks in this agreement.,ready_for_llm,High,0.117,"BEEN ADVISED OF THE POSSIBILITY OR PROBABILITY OF SUCH DAMAGES AND EVEN IF THE REMEDIES OTHERWISE PROVIDED BY THIS AGREEMENT FAIL OF THEIR ESSENTIAL PURPOSE. THE REMEDIES PROVIDED BY THIS AGREEMENT AND THE PROVISIONS OF THIS AGREEMENT ALLOCATE THE RISKS OF THIS AGREEMENT BETWEEN THE PARTIES, SOME OF WHICH MAY BE UNKNOWN OR UNDERMINABLE. THESE LIMITATIONS ARE A MATERIAL INDUCEMENT FOR THE PARTIES TO THIS AGREEMENT TO ENTER INTO THIS AGREEMENT, AND THE PARTIES TO THIS AGREEMENT HAVE RELIED UPON THESE PROVISIONS IN DETERMINING WHETHER OR NOT TO ENTER INTO THIS AGREEMENT."


question,status,answer,latency_seconds
Can you help me plan a trip to Italy?,rejected,"I can only help with contract analysis questions. Please ask about contract clauses, obligations, risks, or agreement terms.",0.0
Can you diagnose my chest pain?,rejected,"I can only help with contract analysis questions. Please ask about contract clauses, obligations, risks, or agreement terms.",0.0


In [0]:
# Test OpenAI API access using the secret stored in Databricks.
# This confirms that GPT-4o-mini can be called without putting the API key in the notebook.

from openai import OpenAI
import time

openai_api_key = dbutils.secrets.get(
    scope="contractscanner",
    key="openai_api_key"
)

client = OpenAI(api_key=openai_api_key)

start_time = time.time()

test_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a contract analysis assistant. Keep answers concise."
        },
        {
            "role": "user",
            "content": "In one sentence, explain what an indemnification clause is."
        }
    ],
    temperature=0.2,
    max_tokens=150
)

latency = round(time.time() - start_time, 3)

print("Model worked.")
print("Latency:", latency, "seconds")
print("Answer:")
print(test_response.choices[0].message.content)

Model worked.
Latency: 4.762 seconds
Answer:
An indemnification clause is a provision in a contract that requires one party to compensate the other for certain damages or losses incurred, typically arising from third-party claims or breaches of the agreement.


Trace(trace_id=tr-4704649fc821daead18a82e108b29b8c)

In [0]:
# Test GPT-4o access.
# This confirms that both required models are available before we build the full evaluation.

start_time = time.time()

test_response_4o = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are a contract analysis assistant. Keep answers concise."
        },
        {
            "role": "user",
            "content": "In one sentence, explain what a termination clause is."
        }
    ],
    temperature=0.2,
    max_tokens=150
)

latency_4o = round(time.time() - start_time, 3)

print("GPT-4o worked.")
print("Latency:", latency_4o, "seconds")
print("Answer:")
print(test_response_4o.choices[0].message.content)

GPT-4o worked.
Latency: 1.07 seconds
Answer:
A termination clause is a provision in a contract that outlines the conditions under which the agreement can be ended by either party.


Trace(trace_id=tr-c5c9d4c6d8bae7929830f49cd68eb199)

## Run GPT-4o and GPT-4o-mini on Contract Questions

This section sends the three prepared contract-analysis questions to both GPT-4o and GPT-4o-mini.

The previous notebook already retrieved the most relevant CUAD contract clause and assigned a basic risk level. Here, the LLMs use that retrieved clause as context and generate the final contract-analysis response.

In [0]:
# This function takes one contract question and its retrieved CUAD clause,
# then asks the selected LLM to answer using only that context.

def call_contract_llm(model_name, question, retrieved_clause, risk_level):
    system_prompt = """
You are ContractScanner, an AI contract analysis assistant for legal and compliance teams.

Your job is to analyze contract clauses using the provided retrieved contract context.
You must:
- Answer only based on the retrieved context.
- Identify possible risks clearly.
- Avoid giving final legal advice.
- Recommend human legal/compliance review when needed.
- Keep the answer professional and concise.
"""

    user_prompt = f"""
User question:
{question}

Retrieved contract clause:
{retrieved_clause}

Initial risk level from tool:
{risk_level}

Write a clear contract analysis response.
"""

    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=500
    )

    latency = round(time.time() - start_time, 3)
    answer = response.choices[0].message.content

    return answer, latency

In [0]:
# Run the 3 contract questions through both GPT-4o and GPT-4o-mini.
# We are not using the rejection examples here because those are already rejected
# by the scope-checking tool and do not need an LLM response.

models_to_compare = ["gpt-4o", "gpt-4o-mini"]

llm_results = []

for _, row in prep_df.iterrows():
    question = row["question"]
    retrieved_clause = row["top_retrieved_clause"]
    risk_level = row["risk_level"]

    for model_name in models_to_compare:
        print(f"Running {model_name} for question: {question}")

        answer, latency = call_contract_llm(
            model_name=model_name,
            question=question,
            retrieved_clause=retrieved_clause,
            risk_level=risk_level
        )

        llm_results.append({
            "question": question,
            "model": model_name,
            "risk_level": risk_level,
            "retrieved_clause": retrieved_clause,
            "answer": answer,
            "latency_seconds": latency
        })

llm_results_df = pd.DataFrame(llm_results)

display(llm_results_df)

Running gpt-4o for question: Does this contract contain risky termination clauses?
Running gpt-4o-mini for question: Does this contract contain risky termination clauses?
Running gpt-4o for question: What indemnification obligations are present?
Running gpt-4o-mini for question: What indemnification obligations are present?
Running gpt-4o for question: Summarize the key risks in this agreement.
Running gpt-4o-mini for question: Summarize the key risks in this agreement.


question,model,risk_level,retrieved_clause,answer,latency_seconds
Does this contract contain risky termination clauses?,gpt-4o,Medium,"32.4. the following Clauses shall survive termination for whatever cause of this Agreement: Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34 inclusive.","The retrieved clause indicates that certain provisions will survive the termination of the agreement, specifically Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, and 30-34. The survival of these clauses suggests that obligations or rights under these sections will continue even after the contract ends. **Possible Risks:** 1. **Ongoing Obligations:** The parties may have continuing obligations that could lead to liabilities or require ongoing compliance efforts. 2. **Unclear Terms:** Without the specific content of these clauses, it's unclear what specific rights or obligations are being extended, which could pose risks if they are unfavorable or burdensome. 3. **Enforcement and Disputes:** Surviving clauses could lead to disputes over their interpretation or enforcement after termination. **Recommendation:** A human legal or compliance professional should review the specific clauses mentioned to assess their content and implications. This will help determine if the survival of these clauses poses significant risks or requires mitigation strategies.",2.973
Does this contract contain risky termination clauses?,gpt-4o-mini,Medium,"32.4. the following Clauses shall survive termination for whatever cause of this Agreement: Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34 inclusive.","The retrieved clause indicates that several specific clauses will survive termination of the agreement, which can pose potential risks. The survival of these clauses may imply ongoing obligations or liabilities for the parties involved even after the contract ends. Key risks include: 1. **Ongoing Obligations**: The surviving clauses may require continued performance or compliance, which could lead to unexpected liabilities. 2. **Unclear Scope**: Without knowing the content of the referenced clauses (4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34), it is difficult to assess the full implications of their survival. 3. **Termination Triggers**: The phrase ""for whatever cause"" suggests that termination could occur under various circumstances, potentially leading to disputes over which obligations remain. Given these considerations, it is advisable to conduct a thorough review of the specific clauses listed to understand their implications fully. A human legal or compliance review is recommended to assess the risks associated with these termination clauses in the context of the entire agreement.",13.427
What indemnification obligations are present?,gpt-4o,High,"The foregoing indemnification obligations are conditioned upon the indemnified party promptly notifying the indemnifying party in writing of the claim, suit or proceeding for which the indemnifying party is obligated under this Section 24, cooperating with, assisting and providing information to, the indemnifying party as reasonably required, and granting the indemnifying party the exclusive right to defend or settle such claim, suit or proceeding.","The indemnification obligations in this contract require the indemnified party to promptly notify the indemnifying party in writing about any claim, suit, or proceeding. Additionally, the indemnified party must cooperate with, assist, and provide necessary information to the indemnifying party. The indemnifying party is granted the exclusive right to defend or settle the claim, suit, or proceeding. The initial risk level is assessed as high, likely due to the potential for significant liability or financial exposure if these obligations are not met. The conditions placed on the indemnified party, such as prompt notification and cooperation, are critical to ensuring that the indemnifying party can effectively manage the defense or settlement of claims. Given the high risk level, it is re

[Trace(trace_id=tr-871b7d542afc1cd6c66abc97e8d631ab), Trace(trace_id=tr-a29f199a50ae8235853d21b489753d7e), Trace(trace_id=tr-9a35732c159fd0139fa99d1a8e30016c), Trace(trace_id=tr-8f5a6a7732cd16732ed676c3dfb5d693), Trace(trace_id=tr-33d9146733b4bc1f6404c9ceeba4aae3), Trace(trace_id=tr-5a1182d6617a5eb56c2213a5eefaca9d)]

## Evaluation Scoring

This section creates simple evaluation scores for the GPT-4o and GPT-4o-mini responses.

The scoring focuses on:
- Relevance to the user question
- Grounding in the retrieved contract clause
- Risk identification
- Clarity
- Human review recommendation

The scores are not meant to replace a legal expert. They are used as a structured way to compare model outputs before human review.

In [0]:
# Simple evaluation function for contract-analysis responses.

def evaluate_contract_response(question, retrieved_clause, risk_level, answer):
    answer_lower = answer.lower()
    question_lower = question.lower()
    clause_lower = retrieved_clause.lower()

    relevance_score = 5 if any(word in answer_lower for word in question_lower.split()[:4]) else 4

    grounding_score = 5 if any(
        keyword in answer_lower
        for keyword in clause_lower.split()[:12]
    ) else 4

    risk_score = 5 if "risk" in answer_lower or risk_level.lower() in answer_lower else 4

    clarity_score = 5 if len(answer.split()) >= 40 and len(answer.split()) <= 180 else 4

    human_review_score = 5 if "legal" in answer_lower or "review" in answer_lower or "counsel" in answer_lower else 4

    average_score = round(
        (
            relevance_score
            + grounding_score
            + risk_score
            + clarity_score
            + human_review_score
        ) / 5,
        2
    )

    return {
        "relevance_score": relevance_score,
        "grounding_score": grounding_score,
        "risk_score": risk_score,
        "clarity_score": clarity_score,
        "human_review_score": human_review_score,
        "average_score": average_score
    }

In [0]:
# Apply the evaluation function to each LLM response.

evaluation_rows = []

for _, row in llm_results_df.iterrows():
    scores = evaluate_contract_response(
        question=row["question"],
        retrieved_clause=row["retrieved_clause"],
        risk_level=row["risk_level"],
        answer=row["answer"]
    )

    evaluation_rows.append({
        "question": row["question"],
        "model": row["model"],
        "risk_level": row["risk_level"],
        "latency_seconds": row["latency_seconds"],
        "answer": row["answer"],
        **scores
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

question,model,risk_level,latency_seconds,answer,relevance_score,grounding_score,risk_score,clarity_score,human_review_score,average_score
Does this contract contain risky termination clauses?,gpt-4o,Medium,2.973,"The retrieved clause indicates that certain provisions will survive the termination of the agreement, specifically Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, and 30-34. The survival of these clauses suggests that obligations or rights under these sections will continue even after the contract ends. **Possible Risks:** 1. **Ongoing Obligations:** The parties may have continuing obligations that could lead to liabilities or require ongoing compliance efforts. 2. **Unclear Terms:** Without the specific content of these clauses, it's unclear what specific rights or obligations are being extended, which could pose risks if they are unfavorable or burdensome. 3. **Enforcement and Disputes:** Surviving clauses could lead to disputes over their interpretation or enforcement after termination. **Recommendation:** A human legal or compliance professional should review the specific clauses mentioned to assess their content and implications. This will help determine if the survival of these clauses poses significant risks or requires mitigation strategies.",5,5,5,5,5,5.0
Does this contract contain risky termination clauses?,gpt-4o-mini,Medium,13.427,"The retrieved clause indicates that several specific clauses will survive termination of the agreement, which can pose potential risks. The survival of these clauses may imply ongoing obligations or liabilities for the parties involved even after the contract ends. Key risks include: 1. **Ongoing Obligations**: The surviving clauses may require continued performance or compliance, which could lead to unexpected liabilities. 2. **Unclear Scope**: Without knowing the content of the referenced clauses (4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34), it is difficult to assess the full implications of their survival. 3. **Termination Triggers**: The phrase ""for whatever cause"" suggests that termination could occur under various circumstances, potentially leading to disputes over which obligations remain. Given these considerations, it is advisable to conduct a thorough review of the specific clauses listed to understand their implications fully. A human legal or compliance review is recommended to assess the risks associated with these termination clauses in the context of the entire agreement.",5,5,5,5,5,5.0
What indemnification obligations are present?,gpt-4o,High,3.855,"The indemnification obligations in this contract require the indemnified party to promptly notify the indemnifying party in writing about any claim, suit, or proceeding. Additionally, the indemnified party must cooperate with, assist, and provide necessary information to the indemnifying party. The indemnifying party is granted the exclusive right to defend or settle the claim, suit, or proceeding. The initial risk level is assessed as high, likely due to the potential for significant liability or financial exposure if these obligations are not met. The conditions placed on the indemnified party, such as prompt notification and cooperation, are critical to ensuring that the indemnifying party can effectively manage the defense or settlement of claims. Given the high risk level, it is recommended that a human legal or compliance professional review these indemnification obligations to ensure that they align with the organization's risk management strategies and contractual standards.",5,5,5,5,5,5.0
What indemnification obligations are present?,gpt-4o-mini,High,7.133,"The indemnification obligations outlined in the retrieved clause include several key conditions that the indemnified party must fulfill: 1. **Prompt Notification**: The indemnified party is required to notify the indemnifying party in writing about any claim, suit, or proceeding in a timely manner. Failure to do so could jeopardize the indemnification rights. 2. *

## ROI Comparison

This section compares GPT-4o and GPT-4o-mini from a business perspective.

The goal is not only to see which model gives better answers, but also which model makes more sense for a real enterprise deployment. ContractScanner would save money by reducing analyst review time, but the LLM cost still matters when the system is used at scale.

In [0]:
# ROI comparison for GPT-4o vs GPT-4o-mini.
# These prices are written as variables so they can be updated if OpenAI pricing changes.

# Current standard OpenAI API prices per 1M tokens.
# Source: OpenAI pricing page.
MODEL_PRICING = {
    "gpt-4o": {
        "input_cost_per_1m": 2.50,
        "output_cost_per_1m": 10.00
    },
    "gpt-4o-mini": {
        "input_cost_per_1m": 0.15,
        "output_cost_per_1m": 0.60
    }
}

# Business assumptions from the original project proposal.
contracts_per_year = 500
hours_saved_per_contract = 1
analyst_hourly_rate = 75

annual_labor_savings = contracts_per_year * hours_saved_per_contract * analyst_hourly_rate

roi_rows = []

for model_name in ["gpt-4o", "gpt-4o-mini"]:
    model_rows = llm_results_df[llm_results_df["model"] == model_name]

    # Because this notebook used chat.completions, token usage was not saved in llm_results_df.
    # For the ROI estimate, we use a conservative average token estimate per contract question.
    estimated_input_tokens_per_contract = 1000
    estimated_output_tokens_per_contract = 350

    input_cost = (
        estimated_input_tokens_per_contract
        / 1_000_000
        * MODEL_PRICING[model_name]["input_cost_per_1m"]
    )

    output_cost = (
        estimated_output_tokens_per_contract
        / 1_000_000
        * MODEL_PRICING[model_name]["output_cost_per_1m"]
    )

    estimated_cost_per_contract = input_cost + output_cost
    estimated_annual_llm_cost = estimated_cost_per_contract * contracts_per_year
    estimated_net_savings = annual_labor_savings - estimated_annual_llm_cost

    avg_latency = round(model_rows["latency_seconds"].mean(), 3)
    avg_score = round(evaluation_df[evaluation_df["model"] == model_name]["average_score"].mean(), 2)

    roi_rows.append({
        "model": model_name,
        "average_eval_score": avg_score,
        "average_latency_seconds": avg_latency,
        "estimated_cost_per_contract_usd": round(estimated_cost_per_contract, 6),
        "estimated_annual_llm_cost_usd": round(estimated_annual_llm_cost, 2),
        "annual_labor_savings_usd": annual_labor_savings,
        "estimated_net_savings_usd": round(estimated_net_savings, 2)
    })

roi_df = pd.DataFrame(roi_rows)

display(roi_df)

model,average_eval_score,average_latency_seconds,estimated_cost_per_contract_usd,estimated_annual_llm_cost_usd,annual_labor_savings_usd,estimated_net_savings_usd
gpt-4o,4.93,3.405,0.006,3.0,37500,37497.0
gpt-4o-mini,4.87,9.184,3.6E-4,0.18,37500,37499.82


In [0]:
# Add the two graceful rejection examples to the final evaluation table.
# These examples show the agent safely rejecting questions outside contract-analysis scope.

rejection_rows = []

for _, row in reject_df.iterrows():
    rejection_rows.append({
        "question": row["question"],
        "model": "scope_rejection_tool",
        "risk_level": "Not applicable",
        "latency_seconds": row["latency_seconds"],
        "answer": row["answer"],
        "relevance_score": 5,
        "grounding_score": 5,
        "risk_score": 5,
        "clarity_score": 5,
        "human_review_score": 5,
        "average_score": 5.0
    })

rejection_eval_df = pd.DataFrame(rejection_rows)

final_evaluation_df = pd.concat(
    [evaluation_df, rejection_eval_df],
    ignore_index=True
)

display(final_evaluation_df)

print("Final evaluation rows:", len(final_evaluation_df))

question,model,risk_level,latency_seconds,answer,relevance_score,grounding_score,risk_score,clarity_score,human_review_score,average_score
Does this contract contain risky termination clauses?,gpt-4o,Medium,2.973,"The retrieved clause indicates that certain provisions will survive the termination of the agreement, specifically Clauses 4.2, 5, 10.2, 20.4, 23.2, 25-28, and 30-34. The survival of these clauses suggests that obligations or rights under these sections will continue even after the contract ends. **Possible Risks:** 1. **Ongoing Obligations:** The parties may have continuing obligations that could lead to liabilities or require ongoing compliance efforts. 2. **Unclear Terms:** Without the specific content of these clauses, it's unclear what specific rights or obligations are being extended, which could pose risks if they are unfavorable or burdensome. 3. **Enforcement and Disputes:** Surviving clauses could lead to disputes over their interpretation or enforcement after termination. **Recommendation:** A human legal or compliance professional should review the specific clauses mentioned to assess their content and implications. This will help determine if the survival of these clauses poses significant risks or requires mitigation strategies.",5,5,5,5,5,5.0
Does this contract contain risky termination clauses?,gpt-4o-mini,Medium,13.427,"The retrieved clause indicates that several specific clauses will survive termination of the agreement, which can pose potential risks. The survival of these clauses may imply ongoing obligations or liabilities for the parties involved even after the contract ends. Key risks include: 1. **Ongoing Obligations**: The surviving clauses may require continued performance or compliance, which could lead to unexpected liabilities. 2. **Unclear Scope**: Without knowing the content of the referenced clauses (4.2, 5, 10.2, 20.4, 23.2, 25-28, 30-34), it is difficult to assess the full implications of their survival. 3. **Termination Triggers**: The phrase ""for whatever cause"" suggests that termination could occur under various circumstances, potentially leading to disputes over which obligations remain. Given these considerations, it is advisable to conduct a thorough review of the specific clauses listed to understand their implications fully. A human legal or compliance review is recommended to assess the risks associated with these termination clauses in the context of the entire agreement.",5,5,5,5,5,5.0
What indemnification obligations are present?,gpt-4o,High,3.855,"The indemnification obligations in this contract require the indemnified party to promptly notify the indemnifying party in writing about any claim, suit, or proceeding. Additionally, the indemnified party must cooperate with, assist, and provide necessary information to the indemnifying party. The indemnifying party is granted the exclusive right to defend or settle the claim, suit, or proceeding. The initial risk level is assessed as high, likely due to the potential for significant liability or financial exposure if these obligations are not met. The conditions placed on the indemnified party, such as prompt notification and cooperation, are critical to ensuring that the indemnifying party can effectively manage the defense or settlement of claims. Given the high risk level, it is recommended that a human legal or compliance professional review these indemnification obligations to ensure that they align with the organization's risk management strategies and contractual standards.",5,5,5,5,5,5.0
What indemnification obligations are present?,gpt-4o-mini,High,7.133,"The indemnification obligations outlined in the retrieved clause include several key conditions that the indemnified party must fulfill: 1. **Prompt Notification**: The indemnified party is required to notify the indemnifying party in writing about any claim, suit, or proceeding in a timely manner. Failure to do so could jeopardize the indemnification rights. 2. *

Final evaluation rows: 8


In [0]:
# Save the final evaluation table so it is available in the repo.

output_path = "agent_outputs/final_llm_evaluation_results.csv"

final_evaluation_df.to_csv(output_path, index=False)

print("Saved final evaluation results to:", output_path)

Saved final evaluation results to: agent_outputs/final_llm_evaluation_results.csv


## Deployment Recommendation

Based on the evaluation and ROI comparison, GPT-4o-mini is the better default model for ContractScanner because it is much cheaper and fast enough for routine contract review questions. It makes the most sense for first-pass review, especially when many contracts need to be processed.

GPT-4o should still be kept as an escalation model. If a question is high-risk, unclear, or receives a low evaluation score, the system can send the same retrieved context to GPT-4o for a stronger second review.

This hybrid approach gives ContractScanner a better balance of cost, speed, and quality. Routine questions stay low-cost, while more complex contract issues can still use the stronger model when needed.

## Final Notes and Deviations from Proposal

The final project followed the original proposal closely. ContractScanner used CUAD-based contract context, retrieved relevant clauses, assigned risk levels, handled out-of-scope rejection examples, compared GPT-4o and GPT-4o-mini, used MLflow tracing, created evaluation scores, explained human review, and included an ROI comparison.

One small change is that this project used prepared CSV outputs from the earlier agent/tool notebook instead of building a full production upload interface. This kept the project focused on the required agent workflow, model comparison, tracing, evaluation, rejection handling, and ROI analysis.

Overall, GPT-4o-mini is recommended as the default model because it is much cheaper and still performed well on the contract questions. GPT-4o should be kept as an escalation model for higher-risk or unclear contract issues.